# Secure Channel: facilitator notebook (the server)

In this lab you play the HTTPS **server**, every student is a **client**, and the public repo is the **internet**. Your key pair is the server's certificate and private key; the fingerprint you show on screen stands in for the certificate check.

Run this on your own laptop. Two files it uses are in `.gitignore` and must stay there:

- `teacher_private_key.pem`: the only thing that can open student feedback. Keep a backup somewhere safe; if it is lost, the feedback cannot be read by anyone.
- `roster.txt`: the student names.

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import json
import secure_channel as sc

## Part 0: One-time setup

Run this cell **once**, before the session. It makes the key pair and refuses to run a second time.

Then publish the public key so students receive it:

```cmd
git add SecureChannel/teacher_public_key.pem
git commit -m "Add facilitator public key"
git push
```

In [ ]:
try:
    print("Created. Fingerprint:", sc.create_teacher_keys())
except FileExistsError as error:
    print(error)

Show this fingerprint on the screen in class. Students compare it with theirs.

In [ ]:
print("Public key fingerprint:", sc.public_key_fingerprint())

## Part 1: Whose key is this?

Copy `roster.example.txt` to `roster.txt` and write one name per line. Then any key or id a student shows you can be traced to its owner.

In [ ]:
try:
    roster = sc.read_roster()
except FileNotFoundError:
    roster = []
    print("roster.txt not found. Copy roster.example.txt to roster.txt first.")

for name in roster:
    key = sc.make_key(name)
    print(sc.student_id(key), key, name)

In [ ]:
key_or_id = input("Paste a student's key or id: ")
print("Owner:", sc.whose_key(key_or_id, roster))

## Part 2: Stage 1 demo

You can open any Stage 1 message knowing only the student's name. So can every student, which is the lesson of the *Break it* exercise.

In [ ]:
name = input("Student name: ")
locked = input("Their locked message: ")
try:
    print(sc.decrypt(locked, sc.make_key(name)))
except ValueError as error:
    print("Could not unlock:", error)

## Part 3: Open the feedback

Merge the students' pull requests on GitHub, then:

```cmd
git pull
```

`id_matches` is `False` when the name written inside the letter does not belong to the id on the envelope.

In [ ]:
letters = []
for envelope in sc.read_envelopes():
    try:
        letters.append(sc.open_envelope(envelope))
    except ValueError as error:
        print("Skipped", envelope.get("student_id"), "-", error)

for number, letter in enumerate(letters):
    flag = "" if letter["id_matches"] else "  (name does not match id!)"
    print(f"[{number}] {letter['sent_at']}  {letter['name']}{flag}")
    print("    ", letter["message"])

## Part 4: Reply

Pick a letter by its number. The reply is locked with that student's session key, so only that student can read it.

In [ ]:
number = int(input("Reply to letter number: "))
text = input("Your reply: ")

reply = sc.seal_reply(text, letters[number])
path = sc.save_envelope(reply, sc.REPLIES_DIR)
print("Saved:", path.name)

Publish the replies:

```cmd
git add SecureChannel/replies
git commit -m "Add replies"
git push
```

Students then run Part 5 of their notebook.